# CIFAR-10 diffusion extraction audit

This notebook downloads CIFAR-10, trains an unconditional diffusion model, and runs two LeakPro attacks through the public `LeakPro` API:

- Carlini et al. Section 5.1, using unconditional generation and an authorized training-reference audit.
- SIDE Algorithm 1 for small diffusion models, using synthetic feature clusters and a time-dependent guidance classifier.

Run this only against models and training data you are authorized to audit. The default `smoke` profile proves that the pipeline runs. One epoch is not enough to reproduce paper-scale memorization or extraction rates. Set `LEAKPRO_CIFAR_PROFILE=demonstration` before starting Jupyter for a longer overfitting experiment.


## Setup

From the repository root, install LeakPro and the notebook-only image dependency with `pip install -e '.[extraction]' torchvision`. CIFAR-10 is downloaded into the system temporary directory; torchvision stores the pretrained ResNet-18 weights in PyTorch's configured hub cache. Set `LEAKPRO_CIFAR_DATA_DIR` to reuse an existing torchvision CIFAR-10 cache. Checkpoints and audit results are also kept outside the repository.


In [ ]:
import json
import os
import sys
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torchvision
import yaml

from leakpro import LeakPro

candidates = [Path.cwd(), *Path.cwd().parents]
example_dir = next(
    (path / 'examples' / 'extraction' for path in candidates if (path / 'examples' / 'extraction' / 'cifar10_diffusion.py').exists()),
    Path.cwd(),
)
if not (example_dir / 'cifar10_diffusion.py').exists():
    raise FileNotFoundError('Run this notebook from the LeakPro checkout or examples/extraction directory.')
sys.path.insert(0, str(example_dir))

from cifar10_diffusion import (
    CIFAR10ExtractionHandler,
    RUN_PROFILES,
    load_cifar10,
    make_adapter,
    make_feature_extractor,
    seed_everything,
    select_device,
    sha256_file,
    sha256_mapping,
    sha256_module_state,
    sha256_tensor,
    train_or_load_target,
)

profile_name = os.getenv('LEAKPRO_CIFAR_PROFILE', 'smoke')
if profile_name not in RUN_PROFILES:
    raise ValueError(f'Unknown profile {profile_name!r}; choose from {sorted(RUN_PROFILES)}.')
profile = RUN_PROFILES[profile_name]
device = select_device()
seed_everything(profile.seed)
work_dir = Path(tempfile.gettempdir()) / 'leakpro_cifar10_extraction' / profile.name
work_dir.mkdir(parents=True, exist_ok=True)
data_dir_value = os.getenv('LEAKPRO_CIFAR_DATA_DIR')
data_dir = Path(data_dir_value).expanduser() if data_dir_value else None
print({'profile': profile.name, 'device': str(device), 'work_dir': str(work_dir), 'data_dir': str(data_dir) if data_dir else 'download'})


## CIFAR-10 target data

The target trains on a deterministic prefix of the CIFAR-10 training split. Those same images are the authorized references used to evaluate near-copy extraction. Images are represented as floating-point BCHW tensors in `[-1, 1]`. No CIFAR-10 test image enters the target or reference set.


In [ ]:
train_dataset, reference_images = load_cifar10(profile, work_dir, data_dir)
assert reference_images.shape == (profile.reference_size, 3, 32, 32)
assert reference_images.dtype == torch.float32
assert torch.isfinite(reference_images).all()
assert float(reference_images.min()) >= -1.0 and float(reference_images.max()) <= 1.0

preview = reference_images[:8].add(1.0).div(2.0)
figure, axes = plt.subplots(1, len(preview), figsize=(12, 2))
for axis, image in zip(axes, preview):
    axis.imshow(image.permute(1, 2, 0))
    axis.axis('off')
figure.suptitle(f'CIFAR-10 target subset, profile={profile.name}')
plt.show()


## Train or load the diffusion target

The target minimizes the standard noise-prediction objective

$$\mathcal{L}=\mathbb{E}_{x_0,t,\epsilon}\left[\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert_2^2\right],$$

with $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$. Sampling uses deterministic DDIM with $\eta=0$. SIDE modifies the predicted noise at every selected reverse step using the classifier gradient already scaled by LeakPro's `guidance_scale`.

A checkpoint is reused only when its stored profile and format version match. Set `LEAKPRO_CIFAR_FORCE_RETRAIN=1` to replace the checkpoint for the selected profile.


In [ ]:
checkpoint_path = work_dir / f'cifar10_ddpm_{profile.name}.pt'
if os.getenv('LEAKPRO_CIFAR_FORCE_RETRAIN') == '1' and checkpoint_path.exists():
    checkpoint_path.unlink()

training_started = time.perf_counter()
model, diffusion, checkpoint_path, epoch_losses = train_or_load_target(
    profile, train_dataset, work_dir, device
)
training_seconds = time.perf_counter() - training_started
checkpoint_hash = sha256_file(checkpoint_path)
print({'checkpoint': str(checkpoint_path), 'sha256': checkpoint_hash, 'seconds': round(training_seconds, 2)})

if epoch_losses:
    plt.figure(figsize=(6, 3))
    plt.plot(epoch_losses)
    plt.xlabel('Epoch')
    plt.ylabel('Noise-prediction MSE')
    plt.title('Target training loss')
    plt.show()


## Probe the target adapter

LeakPro receives the model through `CallableDiffusionAdapter`. The checks below establish the expected shape, range, finiteness, and same-seed determinism before any attack runs. A smoke-profile preview will usually look poor because the model has seen only one epoch.


In [ ]:
adapter = make_adapter(model, diffusion, device)
sample_a = adapter.sample(4, conditions=None, seed=profile.seed + 10)
sample_b = adapter.sample(4, conditions=None, seed=profile.seed + 10)
assert sample_a.shape == (4, 3, 32, 32)
assert torch.isfinite(sample_a).all()
assert float(sample_a.min()) >= -1.0 and float(sample_a.max()) <= 1.0
torch.testing.assert_close(sample_a, sample_b)

figure, axes = plt.subplots(1, 4, figsize=(7, 2))
for axis, image in zip(axes, sample_a.detach().cpu().add(1.0).div(2.0)):
    axis.imshow(image.permute(1, 2, 0))
    axis.axis('off')
figure.suptitle('Unguided DDIM samples')
plt.show()


## Configure SIDE features and the LeakPro handler

SIDE needs frozen semantic features to form surrogate labels. This basic example uses ImageNet-pretrained ResNet-18 embeddings because torchvision can load them directly. The paper's main experiment used SSCD features and a larger classifier, so this is a documented substitution rather than an exact reproduction. Random features are not used as a fallback.


In [ ]:
feature_extractor, feature_transform = make_feature_extractor()
identity_components = {
    'target_checkpoint_sha256': checkpoint_hash,
    'provider_source_sha256': sha256_file(example_dir / 'cifar10_diffusion.py'),
    'side_feature_state_sha256': sha256_module_state(feature_extractor),
    'side_feature_transform': 'resize-224-bilinear-align-corners-false-imagenet-normalization-v1',
    'authorized_references_sha256': sha256_tensor(reference_images),
}
target_fingerprint = f"sha256:{sha256_mapping(identity_components)}"
CIFAR10ExtractionHandler.configure(
    adapter=adapter,
    references=reference_images,
    feature_extractor=feature_extractor,
    feature_transform=feature_transform,
)


## Write and run the audit

Carlini uses the reference-assisted unconditional CIFAR-10 procedure, not the prompt-clique Stable Diffusion procedure. SIDE uses synthetic clustering, time-dependent classifier training, and DDIM classifier guidance. The `smoke` profile lowers SIDE's cohesion threshold so an untrained target can complete the control flow; the `demonstration` profile uses the paper's `0.5` threshold.


In [ ]:
audit_output = work_dir / 'audit'
audit_config = {
    'audit': {
        'attack_type': 'extraction',
        'attack_list': [
            {
                'attack': 'carlini_diffusion',
                'authorized_audit': True,
                'overwrite_results': True,
                'mode': 'unconditional_reference_audit',
                'image_range': 'minus_one_one',
                'num_unconditional_generations': profile.carlini_generations,
                'generation_batch_size': min(64, profile.carlini_generations),
                'reference_neighbors': min(50, profile.reference_size),
                'reference_alpha': 0.5,
                'ratio_threshold': 1.0,
                'verification_l2_threshold': 0.15,
                'max_extractions': 32,
                'distance_block_size': 64,
                'distance_device': 'cpu',
            },
            {
                'attack': 'side',
                'authorized_audit': True,
                'overwrite_results': True,
                'image_range': 'minus_one_one',
                'compute_device': str(device),
                'distance_device': 'cpu',
                'synthetic_samples': profile.side_synthetic_samples,
                'synthetic_batch_size': min(64, profile.side_synthetic_samples),
                'clusters': profile.side_clusters,
                'cohesion_threshold': profile.side_cohesion_threshold,
                'min_cluster_size': 2,
                'kmeans_n_init': 5,
                'classifier_epochs': profile.side_classifier_epochs,
                'classifier_batch_size': 32,
                'classifier_learning_rate': 1e-4,
                'classifier_base_width': 16 if profile.name == 'smoke' else 64,
                'classifier_blocks': [1, 1, 1, 1] if profile.name == 'smoke' else [3, 4, 6, 3],
                'timestep_embedding_dim': 64 if profile.name == 'smoke' else 128,
                'guidance_scale': profile.side_guidance_scale,
                'num_generations': profile.side_generations,
                'generation_batch_size': min(16, profile.side_generations),
                'l2_bands': {'near_copy': {'lower': 0.0, 'upper': 0.3}},
            },
        ],
        'hyper_param_search': False,
        'data_modality': 'image',
        'output_dir': str(audit_output),
    },
    'target': {
        'name': 'cifar10_unconditional_ddim',
        'fingerprint': target_fingerprint,
    },
}
audit_path = work_dir / 'audit.yaml'
audit_path.write_text(yaml.safe_dump(audit_config, sort_keys=False), encoding='utf-8')

audit_started = time.perf_counter()
results = LeakPro(CIFAR10ExtractionHandler, str(audit_path)).run_audit()
audit_seconds = time.perf_counter() - audit_started
assert len(results) == 2
carlini_result, side_result = results
print({'audit_seconds': round(audit_seconds, 2), 'result_ids': [result.id for result in results]})


## Verify and inspect results

Zero Carlini candidates is a valid outcome, especially for the smoke target. It means no generated image passed the adaptive reference-neighborhood ratio. SIDE always returns its guided generations and records their nearest authorized reference when references are supplied. SIDE's stored L2 values use the configured `[-1, 1]` coordinates.


In [ ]:
assert carlini_result.metrics['mode'] == 'unconditional_reference_audit'
assert carlini_result.metrics['images_generated'] == profile.carlini_generations
assert side_result.metrics['images_generated'] == profile.side_generations
assert side_result.metrics['retained_clusters'] >= 2
assert all(torch.isfinite(torch.tensor(side_result.metrics['classifier_epoch_losses'])))
assert side_result.execution_trace[-1]['guidance_calls'] > 0

for result in results:
    result_dir = audit_output / 'results' / result.id
    assert (result_dir / 'result.json').exists()
    assert (result_dir / 'candidates.npz').exists()

print('Carlini metrics:', carlini_result.metrics)
print('SIDE metrics:', side_result.metrics)


In [ ]:
def show_nearest_matches(result, title, maximum=6):
    records = [record for record in result.candidates if record.nearest_reference_index is not None]
    records.sort(key=lambda record: record.nearest_reference_distance)
    records = records[:maximum]
    if not records:
        print(f'{title}: no qualifying candidates')
        return
    figure, axes = plt.subplots(len(records), 2, figsize=(4, 2 * len(records)), squeeze=False)
    for row, record in enumerate(records):
        generated = result.images[record.image_index].detach().cpu()
        reference = reference_images[record.nearest_reference_index].add(1.0).div(2.0)
        axes[row, 0].imshow(generated.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 0].set_title(f'generated, d={record.nearest_reference_distance:.4f}')
        axes[row, 1].imshow(reference.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 1].set_title(f'train #{record.nearest_reference_index}')
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()

show_nearest_matches(carlini_result, 'Carlini qualifying candidates')
show_nearest_matches(side_result, 'SIDE guided samples and nearest training references')


## Save a reproducibility manifest

The manifest records the checkpoint identity, profile, software versions, device, exact audit configuration, and artifact locations. It contains no raw training images.


In [ ]:
manifest = {
    'profile': profile.__dict__,
    'checkpoint': str(checkpoint_path),
    'checkpoint_sha256': checkpoint_hash,
    'target_fingerprint': target_fingerprint,
    'identity_components': identity_components,
    'device': str(device),
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'training_seconds': training_seconds,
    'audit_seconds': audit_seconds,
    'audit_config': audit_config,
    'result_ids': [result.id for result in results],
}
manifest_path = work_dir / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
print(manifest_path)


## Interpretation limits

The `smoke` profile verifies data loading, target training, checkpointing, DDIM sampling, Carlini reference scoring, SIDE clustering, classifier training, guided sampling, persistence, and visualization. It is not extraction-performance evidence.

The `demonstration` profile deliberately trains longer on a smaller subset to make memorization more plausible, but it still does not reproduce either paper's compute. Carlini et al. generated about one million unconditional CIFAR-10 candidates. SIDE reports roughly 2,048 CIFAR-10 training epochs, 10,000 evaluation generations, 100 clusters, cohesion threshold `0.5`, and SSCD features. This notebook keeps the paper-defined attack flow while reducing those budgets and substituting an ImageNet ResNet-18 feature extractor.
